# Notebook 04 — Data Dictionary and Schema Design

## Objective
Document the structure, meaning, data types, constraints, and
relationships of the IPL analytical datasets.

## Source datasets
- matches.csv
- innings.csv
- deliveries.csv

## Scope
1. Dataset schema profiling
2. Column-level data dictionary
3. Primary-key identification
4. Foreign-key identification
5. Entity Relationship Diagram (ERD)
6. Database integrity constraints

## Important
The validated source CSV files must remain unchanged.
All schema documentation will be generated separately.

In [1]:

from pathlib import Path
import pandas as pd
import json
import platform

current_dir = Path.cwd().resolve()

# Detect project root
if (current_dir / "data" / "processed").is_dir():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data" / "processed").is_dir():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError(
        "Project root not detected. Open this notebook "
        "from the project notebooks directory."
    )

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Load datasets
matches_df = pd.read_csv(
    PROCESSED_DATA_DIR / "matches.csv",
    dtype={"match_id": "string"}
)

innings_df = pd.read_csv(
    PROCESSED_DATA_DIR / "innings.csv",
    dtype={"match_id": "string"}
)

deliveries_df = pd.read_csv(
    PROCESSED_DATA_DIR / "deliveries.csv",
    dtype={"match_id": "string"}
)

print("Project root:", PROJECT_ROOT)
print("Operating system:", platform.system())

print("\nDataset dimensions:")
print("Matches:", matches_df.shape)
print("Innings:", innings_df.shape)
print("Deliveries:", deliveries_df.shape)

assert matches_df.shape == (1243, 26)
assert innings_df.shape == (2514, 14)
assert deliveries_df.shape == (295732, 21)

print("\nDATASET LOADING VALIDATION: PASS")


Project root: D:\Git\Personal\ipl-player-performance-v2
Operating system: Windows

Dataset dimensions:
Matches: (1243, 26)
Innings: (2514, 14)
Deliveries: (295732, 21)

DATASET LOADING VALIDATION: PASS


In [2]:

datasets = {
    "matches": matches_df,
    "innings": innings_df,
    "deliveries": deliveries_df,
}

schema_records = []

for table_name, dataframe in datasets.items():

    for column in dataframe.columns:

        sample_values = (
            dataframe[column]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(3)
            .tolist()
        )

        schema_records.append({
            "table_name": table_name,
            "column_name": column,
            "data_type": str(dataframe[column].dtype),
            "total_records": len(dataframe),
            "missing_count": int(dataframe[column].isna().sum()),
            "missing_percentage": round(
                dataframe[column].isna().mean() * 100, 2
            ),
            "unique_values": int(dataframe[column].nunique()),
            "sample_values": json.dumps(sample_values),
        })

schema_profile_df = pd.DataFrame(schema_records)

print("Total tables:", len(datasets))
print("Total columns:", len(schema_profile_df))

for table_name in datasets:
    table_profile = schema_profile_df[
        schema_profile_df["table_name"] == table_name
    ]

    print(f"\n{table_name.upper()}")
    print("Columns:", len(table_profile))
    print("Missing values:", table_profile["missing_count"].sum())

display(schema_profile_df)


Total tables: 3
Total columns: 61

MATCHES
Columns: 26
Missing values: 6227

INNINGS
Columns: 14
Missing values: 2554

DELIVERIES
Columns: 21
Missing values: 0


,table_name,column_name,data_type,total_records,missing_count,missing_percentage,unique_values,sample_values
0,matches,match_id,string,1243,0,0.0,1243,"[""1082591"", ""1082592"", ""1082593""]"
1,matches,source_file,str,1243,0,0.0,1243,"[""1082591.json"", ""1082592.json"", ""1082593.json""]"
2,matches,season,str,1243,0,0.0,19,"[""2017"", ""2018"", ""2019""]"
3,matches,date,str,1243,0,0.0,947,"[""2017-04-05"", ""2017-04-06"", ""2017-04-07""]"
4,matches,match_type,str,1243,0,0.0,1,"[""T20""]"
...,...,...,...,...,...,...,...,...
56,deliveries,is_legal_delivery,bool,295732,0,0.0,2,"[""True"", ""False""]"
57,deliveries,wicket_count,int64,295732,0,0.0,2,"[""0"", ""1""]"
58,deliveries,wicket_kinds,str,295732,0,0.0,11,"[""[]"", ""[\""caught\""]"", ""[\""bowled\""]""]"
59,deliveries,wickets_json,str,295732,0,0.0,11102,"[""[]"", ""[{\""kind\"": \""caught\"", \""player_out\""..."


In [3]:

missing_records = []

for table_name, dataframe in datasets.items():

    for column in dataframe.columns:

        missing_count = int(dataframe[column].isna().sum())

        missing_records.append({
            "table_name": table_name,
            "column_name": column,
            "missing_count": missing_count,
            "missing_percentage": round(
                dataframe[column].isna().mean() * 100, 2
            ),
            "data_type": str(dataframe[column].dtype),
        })

missing_detail_df = pd.DataFrame(missing_records)

# Show only columns containing missing values
missing_columns_df = (
    missing_detail_df[
        missing_detail_df["missing_count"] > 0
    ]
    .sort_values(
        ["table_name", "missing_count"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("Columns with missing values:", len(missing_columns_df))

display(missing_columns_df)

print("\nMissing-value summary by table:")
display(
    missing_columns_df.groupby("table_name")[
        "missing_count"
    ].sum().to_frame("total_missing_values")
)


Columns with missing values: 10


,table_name,column_name,missing_count,missing_percentage,data_type
0,innings,target_runs,1277,50.80,float64
1,innings,target_overs,1277,50.80,float64
2,matches,bowl_out,1243,100.00,float64
3,matches,eliminator,1227,98.71,str
4,matches,method,1220,98.15,str
5,matches,result,1218,97.99,str
6,matches,event_stage,1169,94.05,str
7,matches,event_match_number,74,5.95,float64
8,matches,city_original,51,4.10,str
9,matches,winner,25,2.01,str



Missing-value summary by table:


,total_missing_values
table_name,
innings,2554
matches,6227


In [4]:

for table_name, dataframe in datasets.items():

    print(f"\n{'=' * 70}")
    print(f"TABLE: {table_name.upper()}")
    print("=" * 70)

    for column in dataframe.columns:
        print(
            f"{column:<30} "
            f"{str(dataframe[column].dtype):<12} "
            f"Unique: {dataframe[column].nunique(dropna=True):<8} "
            f"Missing: {dataframe[column].isna().sum()}"
        )



TABLE: MATCHES
match_id                       string       Unique: 1243     Missing: 0
source_file                    str          Unique: 1243     Missing: 0
season                         str          Unique: 19       Missing: 0
date                           str          Unique: 947      Missing: 0
match_type                     str          Unique: 1        Missing: 0
gender                         str          Unique: 1        Missing: 0
team_type                      str          Unique: 1        Missing: 0
overs_limit                    int64        Unique: 1        Missing: 0
venue                          str          Unique: 60       Missing: 0
city                           str          Unique: 37       Missing: 0
city_original                  str          Unique: 37       Missing: 51
city_was_imputed               bool         Unique: 2        Missing: 0
team1                          str          Unique: 19       Missing: 0
team2                          str          Uni

In [5]:

# Matches with missing values
match_missing_columns = matches_df.columns[
    matches_df.isna().any()
].tolist()

print("MATCHES — Missing-value examples")

for column in match_missing_columns:

    print(f"\nColumn: {column}")
    display(
        matches_df.loc[
            matches_df[column].isna(),
            ["match_id", "season", "date", "team1", "team2"]
        ].head(5)
    )

# Innings with missing values
innings_missing_columns = innings_df.columns[
    innings_df.isna().any()
].tolist()

print("\nINNINGS — Missing-value examples")

for column in innings_missing_columns:

    print(f"\nColumn: {column}")
    display(
        innings_df.loc[
            innings_df[column].isna()
        ].head(5)
    )


MATCHES — Missing-value examples

Column: city_original


,match_id,season,date,team1,team2
180,1216493,2020/21,2020-09-20,Delhi Capitals,Kings XI Punjab
182,1216495,2020/21,2020-11-03,Mumbai Indians,Sunrisers Hyderabad
183,1216496,2020/21,2020-09-22,Rajasthan Royals,Chennai Super Kings
185,1216498,2020/21,2020-10-24,Kings XI Punjab,Sunrisers Hyderabad
187,1216500,2020/21,2020-10-09,Delhi Capitals,Rajasthan Royals



Column: winner


,match_id,season,date,team1,team2
33,1082625,2017,2017-04-29,Gujarat Lions,Mumbai Indians
128,1175365,2019,2019-03-30,Kolkata Knight Riders,Delhi Capitals
167,1178424,2019,2019-04-30,Royal Challengers Bangalore,Rajasthan Royals
169,1178426,2019,2019-05-02,Mumbai Indians,Sunrisers Hyderabad
180,1216493,2020/21,2020-09-20,Delhi Capitals,Kings XI Punjab



Column: result


,match_id,season,date,team1,team2
0,1082591,2017,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore
1,1082592,2017,2017-04-06,Rising Pune Supergiant,Mumbai Indians
2,1082593,2017,2017-04-07,Gujarat Lions,Kolkata Knight Riders
3,1082594,2017,2017-04-08,Kings XI Punjab,Rising Pune Supergiant
4,1082595,2017,2017-04-08,Royal Challengers Bangalore,Delhi Daredevils


... (10 outputs left out)


INNINGS — Missing-value examples

Column: target_runs


,match_id,innings_number,batting_team,is_super_over,target_runs,target_overs,overs_recorded,delivery_count,legal_deliveries,total_runs,extras_runs,batter_runs,wicket_records,actual_dismissals
0,1082591,1,Sunrisers Hyderabad,False,NaN,NaN,20,125,120,207,7,200,4,4
2,1082592,1,Mumbai Indians,False,NaN,NaN,20,125,120,184,8,176,8,8
4,1082593,1,Gujarat Lions,False,NaN,NaN,20,122,120,183,4,179,4,4
6,1082594,1,Rising Pune Supergiant,False,NaN,NaN,20,125,120,163,6,157,6,6
8,1082595,1,Royal Challengers Bangalore,False,NaN,NaN,20,124,120,157,6,151,8,8



Column: target_overs


,match_id,innings_number,batting_team,is_super_over,target_runs,target_overs,overs_recorded,delivery_count,legal_deliveries,total_runs,extras_runs,batter_runs,wicket_records,actual_dismissals
0,1082591,1,Sunrisers Hyderabad,False,NaN,NaN,20,125,120,207,7,200,4,4
2,1082592,1,Mumbai Indians,False,NaN,NaN,20,125,120,184,8,176,8,8
4,1082593,1,Gujarat Lions,False,NaN,NaN,20,122,120,183,4,179,4,4
6,1082594,1,Rising Pune Supergiant,False,NaN,NaN,20,125,120,163,6,157,6,6
8,1082595,1,Royal Challengers Bangalore,False,NaN,NaN,20,124,120,157,6,151,8,8


In [6]:

# 1. Match outcome fields
outcome_columns = [
    "winner",
    "result",
    "method",
    "eliminator",
    "bowl_out",
]

print("MATCH OUTCOME FIELD DISTRIBUTIONS")

for column in outcome_columns:
    print(f"\n{column.upper()}")
    display(
        matches_df[column]
        .value_counts(dropna=False)
        .to_frame("record_count")
    )

# 2. Winner and result relationship
print("\nWINNER vs RESULT")

display(
    pd.crosstab(
        matches_df["winner"].fillna("<MISSING>"),
        matches_df["result"].fillna("<MISSING>"),
        margins=True
    )
)

# 3. City imputation verification
print("\nCITY IMPUTATION SUMMARY")

display(
    matches_df.groupby(
        "city_was_imputed",
        dropna=False
    ).agg(
        match_count=("match_id", "count"),
        original_city_missing=("city_original", lambda x: x.isna().sum()),
        mapped_city_missing=("city", lambda x: x.isna().sum()),
    )
)

# 4. Target information by innings
print("\nTARGET FIELDS BY INNINGS")

target_summary = (
    innings_df.groupby(
        ["innings_number", "is_super_over"],
        dropna=False
    )
    .agg(
        innings_count=("match_id", "count"),
        missing_target_runs=("target_runs", lambda x: x.isna().sum()),
        missing_target_overs=("target_overs", lambda x: x.isna().sum()),
        available_target_runs=("target_runs", "count"),
    )
    .reset_index()
)

display(target_summary)

# 5. Check whether the two target columns have matching null patterns
target_null_mismatch = (
    innings_df["target_runs"].isna()
    != innings_df["target_overs"].isna()
).sum()

print("\nTarget null-pattern mismatches:", target_null_mismatch)

assert target_null_mismatch == 0

print("\nMISSING-VALUE SEMANTIC INVESTIGATION COMPLETED")


MATCH OUTCOME FIELD DISTRIBUTIONS

WINNER


,record_count
winner,
Mumbai Indians,155
Chennai Super Kings,148
Kolkata Knight Riders,140
Rajasthan Royals,123
Royal Challengers Bangalore,114
Sunrisers Hyderabad,102
Kings XI Punjab,85
Delhi Daredevils,67
Delhi Capitals,58



RESULT


,record_count
result,
NaN,1218
tie,16
no result,9



METHOD


,record_count
method,
NaN,1220
D/L,23


... (7 outputs left out)

,match_count,original_city_missing,mapped_city_missing
city_was_imputed,,,
False,1192,0,0
True,51,51,0



TARGET FIELDS BY INNINGS


,innings_number,is_super_over,innings_count,missing_target_runs,missing_target_overs,available_target_runs
0,1,False,1243,1243,1243,0
1,2,False,1237,0,0,1237
2,3,True,16,16,16,0
3,4,True,16,16,16,0
4,5,True,1,1,1,0
5,6,True,1,1,1,0



Target null-pattern mismatches: 0

MISSING-VALUE SEMANTIC INVESTIGATION COMPLETED


In [7]:

import pandas as pd

# 1. Outcome reconciliation
result_counts = matches_df["result"].value_counts(dropna=False)

tie_count = int(result_counts.get("tie", 0))
no_result_count = int(result_counts.get("no result", 0))
missing_winner_count = int(matches_df["winner"].isna().sum())

print("OUTCOME RECONCILIATION")
print("Tied matches:", tie_count)
print("No-result matches:", no_result_count)
print("Missing winners:", missing_winner_count)

assert tie_count + no_result_count == missing_winner_count

# 2. D/L method validation
dl_count = int((matches_df["method"] == "D/L").sum())

print("\nD/L matches:", dl_count)

# Check whether D/L records have a winner
dl_missing_winners = matches_df.loc[
    matches_df["method"] == "D/L", "winner"
].isna().sum()

print("D/L records without winner:", dl_missing_winners)

# 3. City imputation validation
assert matches_df.loc[
    matches_df["city_was_imputed"], "city"
].notna().all()

assert matches_df.loc[
    ~matches_df["city_was_imputed"], "city_original"
].notna().all()

print("\nCITY IMPUTATION: PASS")

# 4. Target field validation
target_null_mismatch = (
    innings_df["target_runs"].isna()
    != innings_df["target_overs"].isna()
).sum()

regular_first_innings = innings_df[
    (innings_df["innings_number"] == 1)
    & (~innings_df["is_super_over"])
]

regular_second_innings = innings_df[
    (innings_df["innings_number"] == 2)
    & (~innings_df["is_super_over"])
]

super_over_innings = innings_df[
    innings_df["is_super_over"]
]

print("\nTARGET FIELD VALIDATION")
print("Null-pattern mismatches:", target_null_mismatch)
print("First innings target values:",
      regular_first_innings["target_runs"].notna().sum())
print("Second innings missing targets:",
      regular_second_innings["target_runs"].isna().sum())
print("Super-over target values:",
      super_over_innings["target_runs"].notna().sum())

assert target_null_mismatch == 0
assert regular_first_innings["target_runs"].isna().all()
assert regular_second_innings["target_runs"].notna().all()
assert super_over_innings["target_runs"].isna().all()

print("\nBUSINESS RULE VALIDATION: PASS")


OUTCOME RECONCILIATION
Tied matches: 16
No-result matches: 9
Missing winners: 25

D/L matches: 23
D/L records without winner: 0

CITY IMPUTATION: PASS

TARGET FIELD VALIDATION
Null-pattern mismatches: 0
First innings target values: 0
Second innings missing targets: 0
Super-over target values: 0

BUSINESS RULE VALIDATION: PASS


In [9]:

table_data = {
    "matches": matches_df,
    "innings": innings_df,
    "deliveries": deliveries_df
}

# Total columns across all tables, including repeated names
total_columns = sum(
    len(df.columns) for df in table_data.values()
)

# Unique column names across all tables
all_columns = set().union(
    *(set(df.columns) for df in table_data.values())
)

print("Total tables:", len(table_data))
print("Total columns across tables:", total_columns)
print("Unique column names:", len(all_columns))

# Check descriptions table by table
missing_descriptions = {}

for table_name, df in table_data.items():
    missing = [
        col for col in df.columns
        if col not in column_descriptions
    ]

    missing_descriptions[table_name] = missing

    print(
        f"{table_name}: {len(df.columns)} columns | "
        f"Missing descriptions: {missing}"
    )

# Validate the expected individual table dimensions
assert matches_df.shape[1] == 26
assert innings_df.shape[1] == 14
assert deliveries_df.shape[1] == 21

# Validate total columns across tables
assert total_columns == 61

# Validate description coverage
assert all(
    len(missing) == 0
    for missing in missing_descriptions.values()
), "Some columns do not have descriptions."

print("\nCOLUMN DESCRIPTION COVERAGE: PASS")


Total tables: 3
Total columns across tables: 61
Unique column names: 54
matches: 26 columns | Missing descriptions: []
innings: 14 columns | Missing descriptions: []
deliveries: 21 columns | Missing descriptions: []

COLUMN DESCRIPTION COVERAGE: PASS


In [11]:

# Validate SQL type mappings table by table

table_data = {
    "matches": matches_df,
    "innings": innings_df,
    "deliveries": deliveries_df
}

missing_sql_types = {}

for table_name, df in table_data.items():

    missing = [
        column
        for column in df.columns
        if column not in sql_type_overrides
    ]

    missing_sql_types[table_name] = missing

    print(
        f"{table_name}: "
        f"{len(df.columns)} columns | "
        f"Missing SQL types: {missing}"
    )

total_columns = sum(
    len(df.columns) for df in table_data.values()
)

unique_column_names = set().union(
    *(set(df.columns) for df in table_data.values())
)

print("\nTotal table columns:", total_columns)
print("Unique column names:", len(unique_column_names))
print("SQL type mappings:", len(sql_type_overrides))

assert total_columns == 61

assert all(
    len(missing) == 0
    for missing in missing_sql_types.values()
), "Some columns are missing SQL type definitions."

print("\nSQL TYPE MAPPING VALIDATION: PASS")


matches: 26 columns | Missing SQL types: []
innings: 14 columns | Missing SQL types: []
deliveries: 21 columns | Missing SQL types: []

Total table columns: 61
Unique column names: 54
SQL type mappings: 54

SQL TYPE MAPPING VALIDATION: PASS


In [12]:

table_data = {
    "matches": matches_df,
    "innings": innings_df,
    "deliveries": deliveries_df
}

dictionary_records = []

for table_name, df in table_data.items():

    for column in df.columns:

        dtype = str(df[column].dtype)

        dictionary_records.append({
            "table_name": table_name,
            "column_name": column,
            "description": column_descriptions[column],
            "pandas_dtype": dtype,
            "sql_dtype": sql_type_overrides[column],
            "nullable": bool(df[column].isna().any()),
            "missing_count": int(df[column].isna().sum()),
            "unique_values": int(df[column].nunique(dropna=True)),
            "key_role": (
                "PRIMARY KEY"
                if column in primary_keys[table_name]
                else "FOREIGN KEY"
                if column in foreign_keys[table_name]
                else "NON-KEY"
            )
        })

data_dictionary = pd.DataFrame(dictionary_records)

display(data_dictionary)

print("Total dictionary records:", len(data_dictionary))
print("Unique tables:", data_dictionary["table_name"].nunique())
print("Missing descriptions:",
      data_dictionary["description"].isna().sum())
print("Missing SQL types:",
      data_dictionary["sql_dtype"].isna().sum())

assert len(data_dictionary) == 61
assert data_dictionary["description"].notna().all()
assert data_dictionary["sql_dtype"].notna().all()

print("\nDATA DICTIONARY GENERATION: PASS")


,table_name,column_name,description,pandas_dtype,sql_dtype,nullable,missing_count,unique_values,key_role
0,matches,match_id,Unique identifier assigned to the match.,string,VARCHAR(20),False,0,1243,PRIMARY KEY
1,matches,source_file,Original JSON filename used to extract the match.,str,VARCHAR(100),False,0,1243,NON-KEY
2,matches,season,IPL season in which the match was played.,str,VARCHAR(10),False,0,19,NON-KEY
3,matches,date,Date on which the match was played.,str,DATE,False,0,947,NON-KEY
4,matches,match_type,"Format of the match, such as T20.",str,VARCHAR(20),False,0,1,NON-KEY
...,...,...,...,...,...,...,...,...,...
56,deliveries,is_legal_delivery,Indicates whether the delivery counts as a leg...,bool,BOOLEAN,False,0,2,NON-KEY
57,deliveries,wicket_count,Number of wicket records on the delivery.,int64,SMALLINT,False,0,2,NON-KEY
58,deliveries,wicket_kinds,Serialized or combined dismissal types recorded.,str,VARCHAR(255),False,0,11,NON-KEY
59,deliveries,wickets_json,Serialized wicket details from the source JSON.,str,JSON,False,0,11102,NON-KEY


Total dictionary records: 61
Unique tables: 3
Missing descriptions: 0
Missing SQL types: 0

DATA DICTIONARY GENERATION: PASS


In [14]:

from pathlib import Path
import pandas as pd

DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

dictionary_path = DOCS_DIR / "data_dictionary.csv"

# Export
data_dictionary.to_csv(
    dictionary_path,
    index=False,
    encoding="utf-8"
)

print("Data dictionary exported:", dictionary_path)

# Read back
dictionary_check = pd.read_csv(dictionary_path)

# Validate dimensions
assert dictionary_check.shape == data_dictionary.shape

# Validate table-specific column counts
expected_counts = {
    "matches": 26,
    "innings": 14,
    "deliveries": 21
}

actual_counts = (
    dictionary_check.groupby("table_name")["column_name"]
    .count()
    .to_dict()
)

print("\nColumn counts by table:")
print(actual_counts)

assert actual_counts == expected_counts

# Validate table-column combinations
assert not dictionary_check.duplicated(
    subset=["table_name", "column_name"]
).any()

assert len(dictionary_check) == 61

# Validate completeness
assert dictionary_check["description"].notna().all()
assert dictionary_check["sql_dtype"].notna().all()

print("\nTotal dictionary records:", len(dictionary_check))
print(
    "Unique column names:",
    dictionary_check["column_name"].nunique()
)
print(
    "Unique table-column combinations:",
    dictionary_check[["table_name", "column_name"]]
    .drop_duplicates()
    .shape[0]
)

print("\nCSV READ-BACK VALIDATION: PASS")
print("DATA DICTIONARY EXPORT: PASS")


Data dictionary exported: D:\Git\Personal\ipl-player-performance-v2\docs\data_dictionary.csv

Column counts by table:
{'deliveries': 21, 'innings': 14, 'matches': 26}

Total dictionary records: 61
Unique column names: 54
Unique table-column combinations: 61

CSV READ-BACK VALIDATION: PASS
DATA DICTIONARY EXPORT: PASS


In [15]:

# Explicit database nullability rules

nullable_columns = {
    "matches": [
        "city_original",
        "winner",
        "result",
        "method",
        "eliminator",
        "bowl_out",
        "event_match_number",
        "event_stage"
    ],
    "innings": [
        "target_runs",
        "target_overs"
    ],
    "deliveries": []
}

table_data = {
    "matches": matches_df,
    "innings": innings_df,
    "deliveries": deliveries_df
}

schema_records = []

for table_name, df in table_data.items():

    for column in df.columns:

        is_nullable = column in nullable_columns[table_name]

        schema_records.append({
            "table_name": table_name,
            "column_name": column,
            "observed_missing": int(df[column].isna().sum()),
            "database_nullable": is_nullable
        })

schema_nullability = pd.DataFrame(schema_records)

print("Database nullability by table:")

for table_name in table_data:
    table_schema = schema_nullability[
        schema_nullability["table_name"] == table_name
    ]

    print(
        f"{table_name}: "
        f"{table_schema['database_nullable'].sum()} nullable columns"
    )

# Required key columns must never be nullable
required_keys = {
    "matches": ["match_id"],
    "innings": ["match_id", "innings_number"],
    "deliveries": [
        "match_id",
        "innings_number",
        "over_number",
        "delivery_index_in_over"
    ]
}

for table_name, keys in required_keys.items():
    for column in keys:
        assert column not in nullable_columns[table_name]

assert len(schema_nullability) == 61

print("\nTotal schema fields:", len(schema_nullability))
print("Nullable fields:",
      int(schema_nullability["database_nullable"].sum()))

print("\nDATABASE NULLABILITY VALIDATION: PASS")


Database nullability by table:
matches: 8 nullable columns
innings: 2 nullable columns
deliveries: 0 nullable columns

Total schema fields: 61
Nullable fields: 10

DATABASE NULLABILITY VALIDATION: PASS


In [16]:

key_definitions = {
    "matches": ["match_id"],
    "innings": ["match_id", "innings_number"],
    "deliveries": [
        "match_id",
        "innings_number",
        "over_number",
        "delivery_index_in_over"
    ]
}

for table_name, keys in key_definitions.items():

    df = table_data[table_name]

    duplicate_count = int(
        df.duplicated(subset=keys).sum()
    )

    missing_key_count = int(
        df[keys].isna().any(axis=1).sum()
    )

    print(f"\n{table_name.upper()}")
    print("Primary key:", keys)
    print("Duplicate keys:", duplicate_count)
    print("Rows with missing key values:", missing_key_count)

    assert duplicate_count == 0
    assert missing_key_count == 0

print("\nPRIMARY KEY VALIDATION: PASS")



MATCHES
Primary key: ['match_id']
Duplicate keys: 0
Rows with missing key values: 0

INNINGS
Primary key: ['match_id', 'innings_number']
Duplicate keys: 0
Rows with missing key values: 0

DELIVERIES
Primary key: ['match_id', 'innings_number', 'over_number', 'delivery_index_in_over']
Duplicate keys: 0
Rows with missing key values: 0

PRIMARY KEY VALIDATION: PASS


In [17]:

# Foreign-key integrity validation

# Relationship 1:
# innings.match_id -> matches.match_id

match_ids = set(matches_df["match_id"])

orphan_innings = innings_df[
    ~innings_df["match_id"].isin(match_ids)
].copy()

# Relationship 2:
# deliveries.(match_id, innings_number)
# -> innings.(match_id, innings_number)

innings_keys = pd.MultiIndex.from_frame(
    innings_df[["match_id", "innings_number"]]
)

delivery_keys = pd.MultiIndex.from_frame(
    deliveries_df[["match_id", "innings_number"]]
)

orphan_deliveries = deliveries_df[
    ~delivery_keys.isin(innings_keys)
].copy()

# Report results

print("MATCHES -> INNINGS")
print("Total innings:", len(innings_df))
print("Orphan innings:", len(orphan_innings))

print("\nINNINGS -> DELIVERIES")
print("Total deliveries:", len(deliveries_df))
print("Orphan deliveries:", len(orphan_deliveries))

print("\nForeign-key relationship summary:")

foreign_key_summary = pd.DataFrame([
    {
        "relationship": "innings.match_id -> matches.match_id",
        "child_records": len(innings_df),
        "orphan_records": len(orphan_innings)
    },
    {
        "relationship": (
            "deliveries.(match_id, innings_number) "
            "-> innings.(match_id, innings_number)"
        ),
        "child_records": len(deliveries_df),
        "orphan_records": len(orphan_deliveries)
    }
])

display(foreign_key_summary)

assert len(orphan_innings) == 0
assert len(orphan_deliveries) == 0

print("\nFOREIGN KEY VALIDATION: PASS")


MATCHES -> INNINGS
Total innings: 2514
Orphan innings: 0

INNINGS -> DELIVERIES
Total deliveries: 295732
Orphan deliveries: 0

Foreign-key relationship summary:


,relationship,child_records,orphan_records
0,innings.match_id -> matches.match_id,2514,0
1,"deliveries.(match_id, innings_number) -> innin...",295732,0



FOREIGN KEY VALIDATION: PASS


In [18]:
# Inspect schema metadata before generating MySQL DDL

print("Data dictionary columns:")
print(data_dictionary.columns.tolist())

print("\nData dictionary shape:")
print(data_dictionary.shape)

print("\nData dictionary sample:")
display(data_dictionary.head(10))

print("\nNullability distribution:")
display(
    data_dictionary
    .groupby("table_name")["nullable"]
    .value_counts(dropna=False)
    .to_frame("column_count")
)

Data dictionary columns:
['table_name', 'column_name', 'description', 'pandas_dtype', 'sql_dtype', 'nullable', 'missing_count', 'unique_values', 'key_role']

Data dictionary shape:
(61, 9)

Data dictionary sample:


,table_name,column_name,description,pandas_dtype,sql_dtype,nullable,missing_count,unique_values,key_role
0,matches,match_id,Unique identifier assigned to the match.,string,VARCHAR(20),False,0,1243,PRIMARY KEY
1,matches,source_file,Original JSON filename used to extract the match.,str,VARCHAR(100),False,0,1243,NON-KEY
2,matches,season,IPL season in which the match was played.,str,VARCHAR(10),False,0,19,NON-KEY
3,matches,date,Date on which the match was played.,str,DATE,False,0,947,NON-KEY
4,matches,match_type,"Format of the match, such as T20.",str,VARCHAR(20),False,0,1,NON-KEY
5,matches,gender,Gender category of the competition.,str,VARCHAR(20),False,0,1,NON-KEY
6,matches,team_type,Competition team category.,str,VARCHAR(30),False,0,1,NON-KEY
7,matches,overs_limit,Maximum scheduled overs per innings.,int64,SMALLINT,False,0,1,NON-KEY
8,matches,venue,Cricket stadium where the match was played.,str,VARCHAR(150),False,0,60,NON-KEY
9,matches,city,City associated with the match venue.,str,VARCHAR(100),False,0,37,NON-KEY



Nullability distribution:


column_count
table_name nullable              
deliveries False               21
innings    False               12
           True                 2
matches    False               18
           True                 8

In [19]:

# Inspect key-role metadata

print("Key-role distribution:")
display(
    data_dictionary["key_role"]
    .value_counts(dropna=False)
    .to_frame("column_count")
)

print("\nKey roles by table and column:")
display(
    data_dictionary[
        ["table_name", "column_name", "key_role", "sql_dtype", "nullable"]
    ]
    .sort_values(["table_name", "column_name"])
    .reset_index(drop=True)
)

print("\nComposite primary-key candidates:")
display(
    data_dictionary[
        data_dictionary["key_role"].astype(str).str.contains(
            "primary", case=False, na=False
        )
    ][["table_name", "column_name", "key_role"]]
)


Key-role distribution:


,column_count
key_role,
NON-KEY,54
PRIMARY KEY,7



Key roles by table and column:


,table_name,column_name,key_role,sql_dtype,nullable
0,deliveries,batter,NON-KEY,VARCHAR(100),False
1,deliveries,batter_runs,NON-KEY,INT,False
2,deliveries,batting_team,NON-KEY,VARCHAR(100),False
3,deliveries,bowler,NON-KEY,VARCHAR(100),False
4,deliveries,byes,NON-KEY,SMALLINT,False
...,...,...,...,...,...
56,matches,teams_json,NON-KEY,JSON,False
57,matches,toss_decision,NON-KEY,VARCHAR(20),False
58,matches,toss_winner,NON-KEY,VARCHAR(100),False
59,matches,venue,NON-KEY,VARCHAR(150),False



Composite primary-key candidates:


,table_name,column_name,key_role
0,matches,match_id,PRIMARY KEY
26,innings,match_id,PRIMARY KEY
27,innings,innings_number,PRIMARY KEY
40,deliveries,match_id,PRIMARY KEY
41,deliveries,innings_number,PRIMARY KEY
42,deliveries,over_number,PRIMARY KEY
43,deliveries,delivery_index_in_over,PRIMARY KEY


In [20]:

# Define relational constraints explicitly

primary_keys = {
    "matches": [
        "match_id"
    ],
    "innings": [
        "match_id",
        "innings_number"
    ],
    "deliveries": [
        "match_id",
        "innings_number",
        "over_number",
        "delivery_index_in_over"
    ]
}

foreign_keys = {
    "innings": [
        {
            "columns": ["match_id"],
            "referenced_table": "matches",
            "referenced_columns": ["match_id"]
        }
    ],
    "deliveries": [
        {
            "columns": ["match_id", "innings_number"],
            "referenced_table": "innings",
            "referenced_columns": ["match_id", "innings_number"]
        }
    ]
}

# Validate primary-key metadata against the data dictionary

for table, expected_columns in primary_keys.items():

    actual_columns = (
        data_dictionary.loc[
            (data_dictionary["table_name"] == table)
            & (data_dictionary["key_role"] == "PRIMARY KEY"),
            "column_name"
        ]
        .tolist()
    )

    assert set(actual_columns) == set(expected_columns), (
        f"Primary-key mismatch in {table}: "
        f"expected {expected_columns}, got {actual_columns}"
    )

    print(f"{table}: PRIMARY KEY VALIDATED")

# Validate foreign-key column existence

for child_table, relationships in foreign_keys.items():

    for relationship in relationships:

        parent_table = relationship["referenced_table"]

        child_columns = set(
            data_dictionary.loc[
                data_dictionary["table_name"] == child_table,
                "column_name"
            ]
        )

        parent_columns = set(
            data_dictionary.loc[
                data_dictionary["table_name"] == parent_table,
                "column_name"
            ]
        )

        assert set(relationship["columns"]).issubset(child_columns)
        assert set(relationship["referenced_columns"]).issubset(
            parent_columns
        )

        print(
            f"{child_table} -> {parent_table}: "
            "FOREIGN KEY COLUMNS VALIDATED"
        )

print("\nPRIMARY AND FOREIGN KEY METADATA: PASS")



matches: PRIMARY KEY VALIDATED
innings: PRIMARY KEY VALIDATED
deliveries: PRIMARY KEY VALIDATED
innings -> matches: FOREIGN KEY COLUMNS VALIDATED
deliveries -> innings: FOREIGN KEY COLUMNS VALIDATED

PRIMARY AND FOREIGN KEY METADATA: PASS


In [21]:

from pathlib import Path

# --------------------------------------------------
# 1. Database configuration
# --------------------------------------------------

DATABASE_NAME = "ipl_analytics"

# Explicitly ordered primary keys
primary_keys = {
    "matches": ["match_id"],
    "innings": ["match_id", "innings_number"],
    "deliveries": [
        "match_id",
        "innings_number",
        "over_number",
        "delivery_index_in_over"
    ]
}

# Foreign-key relationships
foreign_keys = {
    "innings": [
        {
            "name": "fk_innings_matches",
            "columns": ["match_id"],
            "referenced_table": "matches",
            "referenced_columns": ["match_id"]
        }
    ],
    "deliveries": [
        {
            "name": "fk_deliveries_innings",
            "columns": ["match_id", "innings_number"],
            "referenced_table": "innings",
            "referenced_columns": [
                "match_id",
                "innings_number"
            ]
        }
    ]
}

# Parent tables must be created before child tables
table_order = ["matches", "innings", "deliveries"]

# --------------------------------------------------
# 2. Validate schema metadata
# --------------------------------------------------

required_columns = {
    "table_name",
    "column_name",
    "sql_dtype",
    "nullable"
}

assert required_columns.issubset(data_dictionary.columns)

assert set(data_dictionary["table_name"]) == set(table_order)

assert not data_dictionary.duplicated(
    ["table_name", "column_name"]
).any()

# --------------------------------------------------
# 3. Generate CREATE TABLE statements
# --------------------------------------------------

ddl_statements = [
    f"CREATE DATABASE IF NOT EXISTS `{DATABASE_NAME}` "
    "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;",
    f"USE `{DATABASE_NAME}`;"
]

for table in table_order:

    table_metadata = data_dictionary[
        data_dictionary["table_name"] == table
    ]

    assert not table_metadata.empty

    definitions = []

    # Column definitions
    for _, row in table_metadata.iterrows():

        column = row["column_name"]
        sql_type = row["sql_dtype"]
        nullable = bool(row["nullable"])

        null_constraint = "NULL" if nullable else "NOT NULL"

        definitions.append(
            f"    `{column}` {sql_type} {null_constraint}"
        )

    # Primary-key constraint
    pk_columns = primary_keys[table]

    assert set(pk_columns).issubset(
        set(table_metadata["column_name"])
    )

    definitions.append(
        "    PRIMARY KEY ("
        + ", ".join(f"`{column}`" for column in pk_columns)
        + ")"
    )

    # Foreign-key constraints
    for fk in foreign_keys.get(table, []):

        child_columns = ", ".join(
            f"`{column}`" for column in fk["columns"]
        )

        parent_columns = ", ".join(
            f"`{column}`" for column in fk["referenced_columns"]
        )

        definitions.append(
            f"    CONSTRAINT `{fk['name']}` "
            f"FOREIGN KEY ({child_columns}) "
            f"REFERENCES `{fk['referenced_table']}` "
            f"({parent_columns}) "
            "ON DELETE RESTRICT "
            "ON UPDATE RESTRICT"
        )

    create_statement = (
        f"CREATE TABLE IF NOT EXISTS `{table}` (\n"
        + ",\n".join(definitions)
        + "\n) ENGINE=InnoDB "
        "DEFAULT CHARSET=utf8mb4 "
        "COLLATE=utf8mb4_unicode_ci;"
    )

    ddl_statements.append(create_statement)

# --------------------------------------------------
# 4. Combine SQL
# --------------------------------------------------

schema_sql = "\n\n".join(ddl_statements)

print(schema_sql)

print("\nDDL GENERATION COMPLETED")


CREATE DATABASE IF NOT EXISTS `ipl_analytics` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;

USE `ipl_analytics`;

CREATE TABLE IF NOT EXISTS `matches` (
    `match_id` VARCHAR(20) NOT NULL,
    `source_file` VARCHAR(100) NOT NULL,
    `season` VARCHAR(10) NOT NULL,
    `date` DATE NOT NULL,
    `match_type` VARCHAR(20) NOT NULL,
    `gender` VARCHAR(20) NOT NULL,
    `team_type` VARCHAR(30) NOT NULL,
    `overs_limit` SMALLINT NOT NULL,
    `venue` VARCHAR(150) NOT NULL,
    `city` VARCHAR(100) NOT NULL,
    `city_original` VARCHAR(100) NULL,
    `city_was_imputed` BOOLEAN NOT NULL,
    `team1` VARCHAR(100) NOT NULL,
    `team2` VARCHAR(100) NOT NULL,
    `teams_json` JSON NOT NULL,
    `toss_winner` VARCHAR(100) NOT NULL,
    `toss_decision` VARCHAR(20) NOT NULL,
    `winner` VARCHAR(100) NULL,
    `result` VARCHAR(30) NULL,
    `method` VARCHAR(30) NULL,
    `eliminator` VARCHAR(100) NULL,
    `bowl_out` INT NULL,
    `player_of_match` VARCHAR(100) NOT NULL,
    `event_name` VAR

In [22]:

# Save generated SQL

SCHEMA_PATH = PROJECT_ROOT / "docs" / "schema.sql"

SCHEMA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

SCHEMA_PATH.write_text(
    schema_sql,
    encoding="utf-8"
)

# Read-back verification
saved_sql = SCHEMA_PATH.read_text(encoding="utf-8")

assert saved_sql == schema_sql

for table in table_order:
    assert f"CREATE TABLE IF NOT EXISTS `{table}`" in saved_sql

for fk in [
    "fk_innings_matches",
    "fk_deliveries_innings"
]:
    assert fk in saved_sql

print("SQL file:", SCHEMA_PATH)
print("File size:", SCHEMA_PATH.stat().st_size, "bytes")
print("Tables generated:", len(table_order))
print("Foreign keys generated:", 2)
print("Read-back verification: PASS")

print("\nMYSQL DDL EXPORT: PASS")


SQL file: D:\Git\Personal\ipl-player-performance-v2\docs\schema.sql
File size: 3200 bytes
Tables generated: 3
Foreign keys generated: 2
Read-back verification: PASS

MYSQL DDL EXPORT: PASS


In [23]:

import json
import re
import pandas as pd

# Map actual DataFrames
table_frames = {
    "matches": matches_df,
    "innings": innings_df,
    "deliveries": deliveries_df
}

# --------------------------------------------------
# 1. VARCHAR length validation
# --------------------------------------------------

varchar_issues = []

for _, row in data_dictionary.iterrows():

    table = row["table_name"]
    column = row["column_name"]
    sql_type = str(row["sql_dtype"])

    match = re.fullmatch(
        r"VARCHAR\((\d+)\)",
        sql_type,
        flags=re.IGNORECASE
    )

    if not match:
        continue

    max_allowed = int(match.group(1))
    series = table_frames[table][column]

    actual_lengths = series.dropna().astype(str).str.len()
    actual_max = int(actual_lengths.max()) if len(actual_lengths) else 0

    if actual_max > max_allowed:
        varchar_issues.append({
            "table": table,
            "column": column,
            "allowed": max_allowed,
            "actual_max": actual_max
        })

print("VARCHAR violations:", len(varchar_issues))

if varchar_issues:
    display(pd.DataFrame(varchar_issues))

# --------------------------------------------------
# 2. JSON validation
# --------------------------------------------------

json_columns = (
    data_dictionary.loc[
        data_dictionary["sql_dtype"].str.upper() == "JSON",
        ["table_name", "column_name"]
    ]
)

json_issues = []

for _, row in json_columns.iterrows():

    table = row["table_name"]
    column = row["column_name"]

    for index, value in table_frames[table][column].items():

        if pd.isna(value):
            continue

        try:
            json.loads(value)
        except (TypeError, ValueError):
            json_issues.append({
                "table": table,
                "column": column,
                "index": index,
                "value": str(value)[:100]
            })

print("\nJSON columns checked:", len(json_columns))
print("Invalid JSON values:", len(json_issues))

if json_issues:
    display(pd.DataFrame(json_issues).head(20))

# --------------------------------------------------
# 3. DATE validation
# --------------------------------------------------

date_issues = []

for _, row in data_dictionary.iterrows():

    if str(row["sql_dtype"]).upper() != "DATE":
        continue

    table = row["table_name"]
    column = row["column_name"]

    parsed = pd.to_datetime(
        table_frames[table][column],
        errors="coerce"
    )

    invalid_count = int(parsed.isna().sum())

    date_issues.append({
        "table": table,
        "column": column,
        "invalid_dates": invalid_count
    })

print("\nDATE validation:")
display(pd.DataFrame(date_issues))

# --------------------------------------------------
# 4. Final audit
# --------------------------------------------------

assert len(varchar_issues) == 0, "VARCHAR length violation found"
assert len(json_issues) == 0, "Invalid JSON found"
assert all(
    item["invalid_dates"] == 0
    for item in date_issues
), "Invalid DATE values found"

print("\nSQL DATA COMPATIBILITY AUDIT: PASS")


VARCHAR violations: 0

JSON columns checked: 3
Invalid JSON values: 0

DATE validation:


,table,column,invalid_dates
0,matches,date,0



SQL DATA COMPATIBILITY AUDIT: PASS


In [24]:

import pandas as pd
from decimal import Decimal, InvalidOperation

# Signed MySQL numeric limits
numeric_limits = {
    "SMALLINT": (-32768, 32767),
    "INT": (-2147483648, 2147483647)
}

numeric_issues = []
nullability_issues = []

for _, row in data_dictionary.iterrows():

    table = row["table_name"]
    column = row["column_name"]
    sql_type = str(row["sql_dtype"]).upper()
    nullable = bool(row["nullable"])

    series = table_frames[table][column]

    # 1. Verify NOT NULL constraints
    if not nullable and series.isna().any():
        nullability_issues.append({
            "table": table,
            "column": column,
            "missing_count": int(series.isna().sum())
        })

    # 2. Verify integer ranges
    if sql_type in numeric_limits:

        lower, upper = numeric_limits[sql_type]
        values = pd.to_numeric(series.dropna(), errors="coerce")

        invalid = values.isna() | (values < lower) | (values > upper)

        if invalid.any():
            numeric_issues.append({
                "table": table,
                "column": column,
                "sql_type": sql_type,
                "invalid_count": int(invalid.sum())
            })

    # 3. Verify BOOLEAN values
    elif sql_type == "BOOLEAN":

        values = pd.to_numeric(series.dropna(), errors="coerce")
        invalid = values.isna() | ~values.isin([0, 1])

        if invalid.any():
            numeric_issues.append({
                "table": table,
                "column": column,
                "sql_type": sql_type,
                "invalid_count": int(invalid.sum())
            })

    # 4. Verify DECIMAL precision and scale
    elif sql_type.startswith("DECIMAL("):

        precision, scale = map(
            int,
            sql_type.replace("DECIMAL(", "").replace(")", "").split(",")
        )

        max_abs = Decimal(10) ** (precision - scale) - (
            Decimal(10) ** (-scale)
        )

        for value in series.dropna():

            try:
                decimal_value = Decimal(str(value))
                exponent = max(0, -decimal_value.as_tuple().exponent)

                if abs(decimal_value) > max_abs or exponent > scale:
                    raise ValueError("Outside DECIMAL definition")

            except (InvalidOperation, ValueError):
                numeric_issues.append({
                    "table": table,
                    "column": column,
                    "sql_type": sql_type,
                    "invalid_value": str(value)
                })

print("Numeric compatibility issues:", len(numeric_issues))
print("NOT NULL violations:", len(nullability_issues))

if numeric_issues:
    display(pd.DataFrame(numeric_issues))

if nullability_issues:
    display(pd.DataFrame(nullability_issues))

assert not numeric_issues
assert not nullability_issues

print("\nNUMERIC AND NULLABILITY VALIDATION: PASS")


Numeric compatibility issues: 0
NOT NULL violations: 0

NUMERIC AND NULLABILITY VALIDATION: PASS


In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "mysql_import"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "matches": {
        "boolean_columns": ["city_was_imputed"]
    },
    "innings": {
        "boolean_columns": ["is_super_over"]
    },
    "deliveries": {
        "boolean_columns": ["is_legal_delivery"]
    }
}

for table, settings in tables.items():

    df = pd.read_csv(SOURCE_DIR / f"{table}.csv")

    assert df.columns.is_unique, (
        f"Duplicate column names in {table}"
    )

    for column in settings["boolean_columns"]:

        values = df[column].astype(str).str.strip().str.lower()

        converted = values.map({
            "true": 1,
            "false": 0,
            "1": 1,
            "0": 0
        })

        assert converted.notna().all(), (
            f"Unexpected Boolean values in {table}.{column}"
        )

        df[column] = converted.astype("int8")

    output_file = OUTPUT_DIR / f"{table}.csv"

    df.to_csv(output_file, index=False)

    # Read-back validation
    check = pd.read_csv(output_file)

    assert check.shape == df.shape
    assert check.columns.is_unique

    print(f"\n{table.upper()}")
    print("Rows:", len(check))
    print("Columns:", len(check.columns))
    print("Boolean columns:", settings["boolean_columns"])
    print("Export:", output_file)
    print("Read-back: PASS")

print("\nALL THREE MYSQL IMPORT FILES GENERATED")


MATCHES
Rows: 1243
Columns: 26
Boolean columns: ['city_was_imputed']
Export: D:\Git\Personal\ipl-player-performance-v2\data\interim\mysql_import\matches.csv
Read-back: PASS

INNINGS
Rows: 2514
Columns: 14
Boolean columns: ['is_super_over']
Export: D:\Git\Personal\ipl-player-performance-v2\data\interim\mysql_import\innings.csv
Read-back: PASS

DELIVERIES
Rows: 295732
Columns: 21
Boolean columns: ['is_legal_delivery']
Export: D:\Git\Personal\ipl-player-performance-v2\data\interim\mysql_import\deliveries.csv
Read-back: PASS

ALL THREE MYSQL IMPORT FILES GENERATED
